# Data quality & validation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def rate(x): return float(np.mean(np.asarray(x)))
def nearest_predict(train_x, train_y, test_x):
    train_x=np.asarray(train_x); train_y=np.asarray(train_y); preds=[]
    for t in np.asarray(test_x):
        preds.append(train_y[int(np.argmin(np.abs(train_x-t)))])
    return np.asarray(preds)
print('setup ok')

## Executable expectations for data
Data validation turns assumptions into checks. These examples validate schema, ranges, missingness, distribution shift, and train-serving skew with small executable measurements.

In [ ]:
# 1) Schema checks catch wrong shapes before models see them
X=np.ones((5,3)); expected_cols=3
plt.figure(figsize=(6,3)); plt.imshow(X,aspect='auto'); plt.title(f'rows x cols = {X.shape}')
assert X.shape[1]==expected_cols

In [ ]:
# 2) Range checks catch impossible values
score=np.array([.5,.6,1.7,.8]); bad=(score<0)|(score>1)
plt.figure(figsize=(6,3)); plt.bar(range(len(score)),score,color=['tab:red' if b else 'tab:blue' for b in bad]); plt.axhline(1,c='k'); plt.title('one probability exceeds 1')
assert bad.sum()==1

In [ ]:
# 3) Missingness thresholds turn silent decay into a failing check
M=np.array([[1,1,0],[1,0,0],[1,1,1],[0,0,0]]); miss=1-M.mean(axis=0)
plt.figure(figsize=(6,3)); plt.bar(['a','b','c'],miss); plt.axhline(.4,c='r',ls='--'); plt.title('feature c violates missingness budget')
assert np.array_equal(miss>.4,np.array([False,True,True]))

In [ ]:
# 4) PSI measures distribution shift between train and serving bins
p=np.array([.5,.3,.2]); q=np.array([.3,.4,.3]); psi=np.sum((q-p)*np.log(q/p))
plt.figure(figsize=(6,3)); plt.bar(['low','mid','high'],q-p); plt.title(f'PSI = {psi:.3f}')
assert abs(psi-0.1714798428)<1e-10

In [ ]:
# 5) Train-serving skew compares the same feature in two pipelines
train=np.array([1.,2.,3.,4.]); serve=np.array([1.1,1.9,3.2,3.7]); diff=np.abs(train-serve)
plt.figure(figsize=(6,3)); plt.bar(range(4),diff); plt.axhline(.25,c='r',ls='--'); plt.title('largest row-level skew')
assert abs(diff.max()-0.3)<1e-12